## 03 - Training

**Imports and paths**

In [ ]:
import itertools
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
import yaml
from dotenv import load_dotenv
from ruamel.yaml import YAML
from scipy import stats
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

import wandb

PROJECT_ROOT = Path("..").resolve()
CONFIG_PATH = PROJECT_ROOT / "config.yml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

TARGET = cfg["data"]["target"]
FEATURES = cfg["features"]
ENCODERS_DIR = PROJECT_ROOT / cfg["artifacts"]["encoders_dir"]
MODELS_DIR = PROJECT_ROOT / cfg["artifacts"]["models_dir"]
METRICS_PATH = PROJECT_ROOT / cfg["artifacts"]["metrics_path"]
FOLD_SCORES_PATH = PROJECT_ROOT / cfg["artifacts"]["fold_scores_path"]
SIGNIFICANCE_PATH = PROJECT_ROOT / cfg["artifacts"]["significance_path"]

# One versioned file for all preprocessing objects and one for all trained models
VERSION = cfg["experiment"]["version"]
PREPROCESSORS_PATH = ENCODERS_DIR / f"preprocessors_{VERSION}.joblib"
MODELS_PATH = MODELS_DIR / f"models_{VERSION}.joblib"

# Shared W&B settings; run files go to <project root>/wandb instead of the notebook folder
WANDB_SETTINGS = {
    "project": cfg["wandb"]["project"],
    "mode": cfg["wandb"]["mode"],
    "dir": PROJECT_ROOT / cfg["wandb"]["dir"],
    "group": VERSION,
}

**3.1 Load cleaned dataset**

In [2]:
# Identifiers and the raw visit date are dropped; they carry no generalisable signal for a new patient
df = pd.read_csv(PROJECT_ROOT / cfg["data"]["cleaned_path"])
df = df.drop(columns=cfg["data"]["drop_columns"])
print(df.shape)
df.head()

(4974, 14)


,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,25.0,F,Boston,22.000000,120/80,83.0,190.0,N,Y,none,30.0,A00,Patient feeling fine 🙂,0
1,25.0,M,Los Angeles,30.100000,120/80,85.0,125.0,N,N,none,14.0,I10,Patient feeling fine 🙂,1
2,20.0,F,New York,22.000000,120/80,63.0,125.0,N,Ex,none,30.0,E11.9,No Notes,0
3,25.0,F,Boston,23.075909,120/80,85.0,250.0,N,Ex,"aspirin,metformin",30.0,Unknown,No Notes,0
4,20.0,F,Chicago,21.500000,120/80,85.0,250.0,N,Ex,none,30.0,A00,"BP high, needs recheck",1


**3.2 Train/test split**

In [3]:
# Stratify to keep the same class balance in both splits; the fixed seed makes the split reproducible
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg["data"]["test_size"],
    stratify=y,
    random_state=cfg["data"]["random_state"],
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (3979, 13) | Test: (995, 13)


**3.3 Fit encoders and scaler on train**

In [ ]:
# Every encoder is fitted on training rows only, so no information from the test set leaks into the features.
# Wrapped in a function because cross-validation (3.7) refits the encoders inside every fold.
def fit_encoders(X_fit: pd.DataFrame) -> dict:
    # Nominal categories have no order, so each value gets its own 0/1 column; unseen values at inference become all zeros
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    one_hot_encoder.fit(X_fit[FEATURES["one_hot"]])

    # Blood_Pressure has hundreds of distinct readings, too many for one-hot, so each reading is encoded by how often it occurs
    frequency_maps = {col: X_fit[col].value_counts(normalize=True).to_dict() for col in FEATURES["frequency"]}

    # Follow_Up has two ordered values (14, 30 days); OrdinalEncoder is the feature-side equivalent of LabelEncoder
    ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    ordinal_encoder.fit(X_fit[FEATURES["ordinal"]])

    # Put numeric features on one scale so distance and weight based models (SVM, LR) are not dominated by large values
    scaler = StandardScaler()
    scaler.fit(X_fit[FEATURES["scale"]])

    return {
        "one_hot_encoder": one_hot_encoder,
        "frequency_maps": frequency_maps,
        "ordinal_encoder": ordinal_encoder,
        "standard_scaler": scaler,
    }


encoders = fit_encoders(X_train)

**3.4 Transform train and test**

In [ ]:
def transform_features(X: pd.DataFrame, encoders: dict) -> pd.DataFrame:
    """Apply fitted encoders and scaler; inference must apply exactly the same steps."""
    scaler = encoders["standard_scaler"]
    ordinal_encoder = encoders["ordinal_encoder"]
    one_hot_encoder = encoders["one_hot_encoder"]
    frequency_maps = encoders["frequency_maps"]

    scaled = pd.DataFrame(scaler.transform(X[FEATURES["scale"]]), columns=FEATURES["scale"], index=X.index)
    ordinal = pd.DataFrame(ordinal_encoder.transform(X[FEATURES["ordinal"]]), columns=FEATURES["ordinal"], index=X.index)
    # Readings never seen in training get frequency 0
    frequency = pd.DataFrame(
        {f"{col}_freq": X[col].map(frequency_maps[col]).fillna(0.0) for col in FEATURES["frequency"]},
        index=X.index,
    )
    one_hot = pd.DataFrame(
        one_hot_encoder.transform(X[FEATURES["one_hot"]]),
        columns=one_hot_encoder.get_feature_names_out(),
        index=X.index,
    )
    return pd.concat([scaled, ordinal, frequency, one_hot], axis=1)


X_train_enc = transform_features(X_train, encoders)
X_test_enc = transform_features(X_test, encoders)
print(f"Encoded train: {X_train_enc.shape} | Encoded test: {X_test_enc.shape}")
X_train_enc.head()

**3.5 Save preprocessors**

In [ ]:
ENCODERS_DIR.mkdir(parents=True, exist_ok=True)

# All fitted preprocessing objects in one file, so inference always loads a matching set
preprocessors = {
    **encoders,
    # Column order the models were trained on; inference input must match it exactly
    "feature_columns": list(X_train_enc.columns),
}
joblib.dump(preprocessors, PREPROCESSORS_PATH)
print(f"Saved {list(preprocessors)} to {PREPROCESSORS_PATH.name}")

**3.6 Weights & Biases login**

In [7]:
# Load the API key from .env (ignored by git) so it is never hard-coded in the notebook or config
load_dotenv(PROJECT_ROOT / cfg["wandb"]["env_file"])

# Team/username also comes from .env; read it only after the file is loaded
WANDB_SETTINGS["entity"] = os.getenv(cfg["wandb"]["entity_env"])

if cfg["wandb"]["mode"] == "online":
    api_key = os.getenv(cfg["wandb"]["api_key_env"])
    if not api_key:
        raise ValueError(f"Set {cfg['wandb']['api_key_env']} in {cfg['wandb']['env_file']} or change wandb.mode in config.yml")
    wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/nipuna/.netrc
wandb: Currently logged in as: tennakoon1997 (tennakoon1997-nipunadev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


**3.7 Cross-validate, train and evaluate models**

In [ ]:
MODEL_CLASSES = {
    "random_forest": RandomForestClassifier,
    "logistic_regression": LogisticRegression,
    "svm": SVC,
    "xgboost": XGBClassifier,
    # No-skill reference: predicts class frequencies only, so it scores ROC-AUC 0.5 on every fold
    "baseline": DummyClassifier,
}
BASELINE = "baseline"
N_SPLITS = cfg["cv"]["n_splits"]
N_REPEATS = cfg["cv"]["n_repeats"]


def evaluate(model, X: pd.DataFrame, y: pd.Series) -> dict:
    pred = model.predict(X)
    # ROC-AUC only needs a ranking score; SVC's predict_proba comes from an internal CV calibration
    # that can disagree with its own predictions, so prefer decision_function when the model has one
    scores = model.decision_function(X) if hasattr(model, "decision_function") else model.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
    }


def cross_validate(name: str, params: dict, X: pd.DataFrame, y: pd.Series) -> list[dict]:
    """Repeated stratified k-fold on the training split only; the test set is never touched here.

    Each repeat reshuffles the rows into new folds. The fixed seed gives every model exactly the same
    folds, so their scores can be compared fold by fold (3.10). Encoders and scaler are refitted inside
    every fold on that fold's training part, so the held-out fold is as unseen as real new data would be.
    Scores are returned in order: repeat 1 folds 1..k, repeat 2 folds 1..k, ...
    """
    folds = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=cfg["data"]["random_state"])
    fold_metrics = []
    for fit_idx, val_idx in folds.split(X, y):
        X_fit, X_val = X.iloc[fit_idx], X.iloc[val_idx]
        y_fit, y_val = y.iloc[fit_idx], y.iloc[val_idx]

        fold_encoders = fit_encoders(X_fit)
        model = MODEL_CLASSES[name](**params)
        model.fit(transform_features(X_fit, fold_encoders), y_fit)
        fold_metrics.append(evaluate(model, transform_features(X_val, fold_encoders), y_val))
    return fold_metrics


def summarize_folds(fold_metrics: list[dict]) -> dict:
    """Mean and standard deviation of each metric across folds."""
    summary = {}
    for metric in fold_metrics[0]:
        values = [fold[metric] for fold in fold_metrics]
        summary[f"{metric}_mean"] = float(np.mean(values))
        summary[f"{metric}_std"] = float(np.std(values))
    return summary


def fold_positions(n_scores: int) -> list[tuple[int, int]]:
    """(repeat, fold) number of each score, in the order cross_validate returns them."""
    return [(i // N_SPLITS + 1, i % N_SPLITS + 1) for i in range(n_scores)]


def log_folds(run, fold_metrics: list[dict], cv_summary: dict) -> None:
    """Per-fold scores as a chart and a table, plus mean/std in the run summary for comparing models."""
    run.define_metric("split")
    run.define_metric("cv/*", step_metric="split")
    fold_table = wandb.Table(columns=["repeat", "fold", *fold_metrics[0]])
    for split, (metrics, (repeat, fold)) in enumerate(zip(fold_metrics, fold_positions(len(fold_metrics))), start=1):
        run.log({"split": split, **{f"cv/{k}": v for k, v in metrics.items()}})
        fold_table.add_data(repeat, fold, *metrics.values())
    run.log({"cv/folds": fold_table})
    for key, value in cv_summary.items():
        run.summary[f"cv/{key}"] = value

In [ ]:
results = []
trained_models = {}
fold_scores = {}  # every model's per-fold scores, kept for the statistical comparison (3.10)

for name, spec in cfg["models"].items():
    # Repeated cross-validation estimates how good and how stable the model is; it drives model selection (3.10-3.11)
    fold_metrics = cross_validate(name, spec["params"], X_train, y_train)
    fold_scores[name] = fold_metrics
    cv_summary = summarize_folds(fold_metrics)

    # Final model is refitted on the whole training split; the test set is scored once, for reporting
    model = MODEL_CLASSES[name](**spec["params"])
    model.fit(X_train_enc, y_train)

    train_metrics = evaluate(model, X_train_enc, y_train)
    test_metrics = evaluate(model, X_test_enc, y_test)
    trained_models[name] = {"version": spec["version"], "model": model}

    # One W&B run per model, grouped by experiment version: fold scores, CV mean/std and final train/test metrics
    run = wandb.init(
        **WANDB_SETTINGS,
        name=f"{name}-v{spec['version']}",
        job_type="train",
        config={
            "model": name,
            "model_version": spec["version"],
            "params": spec["params"],
            "features": FEATURES,
            "n_encoded_features": X_train_enc.shape[1],
            "train_rows": len(X_train_enc),
            "test_rows": len(X_test_enc),
            "cv_folds": N_SPLITS,
            "cv_repeats": N_REPEATS,
            "sklearn_version": sklearn.__version__,
            "xgboost_version": xgboost.__version__,
        },
    )
    run.log({
        **{f"train/{k}": v for k, v in train_metrics.items()},
        **{f"test/{k}": v for k, v in test_metrics.items()},
        "test/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_test.tolist(),
            preds=model.predict(X_test_enc).tolist(),
            class_names=["no disease", "disease"],
        ),
    })
    log_folds(run, fold_metrics, cv_summary)
    run.finish()

    results.append({
        "model": name,
        "version": spec["version"],
        **{f"cv_{k}": v for k, v in cv_summary.items()},
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    print(
        f"{name}: CV ROC-AUC {cv_summary['roc_auc_mean']:.3f} ± {cv_summary['roc_auc_std']:.3f}, "
        f"test ROC-AUC {test_metrics['roc_auc']:.3f}"
    )

**3.8 Save models**

In [10]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# All trained models in one versioned file; each entry keeps its own model version from config.yml
joblib.dump(trained_models, MODELS_PATH)
print(f"Saved {list(trained_models)} to {MODELS_PATH.name}")

# Log both files as W&B artifacts, tagged with the experiment version, so each run is traceable to its exact files
run = wandb.init(**WANDB_SETTINGS, name=f"artifacts-{VERSION}", job_type="upload")
for path, artifact_type in [(PREPROCESSORS_PATH, "preprocessor"), (MODELS_PATH, "model")]:
    artifact = wandb.Artifact(path.stem.removesuffix(f"_{VERSION}"), type=artifact_type, metadata={"experiment_version": VERSION})
    artifact.add_file(str(path))
    run.log_artifact(artifact, aliases=[VERSION])
run.finish()

Saved ['random_forest', 'logistic_regression', 'svm', 'xgboost'] to models_v1.joblib


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


**3.9 Model comparison**

In [11]:
comparison = pd.DataFrame(results).set_index("model").round(4)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(METRICS_PATH)
comparison

,version,train_accuracy,train_precision,train_recall,train_f1,train_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
model,,,,,,,,,,,
random_forest,1.0.0,0.8110,0.8122,0.8040,0.8081,0.8982,0.5085,0.5034,0.4553,0.4781,0.5124
logistic_regression,1.0.0,0.5147,0.5109,0.4540,0.4808,0.5283,0.4864,0.4782,0.4228,0.4488,0.4840
svm,1.0.0,0.6725,0.6709,0.6638,0.6673,0.7343,0.5025,0.4969,0.4939,0.4954,0.4921
xgboost,1.0.0,0.7175,0.7155,0.7125,0.7140,0.8089,0.4985,0.4927,0.4817,0.4872,0.4821


**3.10 Statistical comparison**

In [ ]:
def corrected_resampled_ttest(scores_a, scores_b, test_train_ratio: float) -> tuple[float, float, float]:
    """Paired t-test on per-fold scores, corrected for overlapping training sets (Nadeau & Bengio, 2003).

    CV folds share most of their training rows, so their scores are correlated and the ordinary
    paired t-test overstates significance. The correction inflates the variance by the held-out to
    training size ratio (1/4 for 5-fold). Returns (mean difference a - b, t statistic, two-sided p).
    """
    diff = np.asarray(scores_a, dtype=float) - np.asarray(scores_b, dtype=float)
    n = len(diff)
    mean = float(diff.mean())
    variance = float(diff.var(ddof=1))
    if variance == 0:
        # Identical difference on every fold (e.g. two constant baselines): no evidence either way unless it is non-zero
        return mean, 0.0 if mean == 0 else float(np.copysign(np.inf, mean)), 1.0 if mean == 0 else 0.0
    t_stat = mean / np.sqrt((1 / n + test_train_ratio) * variance)
    p_value = 2 * stats.t.sf(abs(t_stat), df=n - 1)
    return mean, float(t_stat), float(p_value)


def holm_adjust(p_values) -> np.ndarray:
    """Holm-Bonferroni adjustment: keeps the chance of any false 'significant' result at alpha across all pairs."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        running_max = max(running_max, min(1.0, (m - rank) * p[idx]))
        adjusted[idx] = running_max
    return adjusted

In [ ]:
# The no-skill baseline is cross-validated on exactly the same folds as the models
fold_scores[BASELINE] = cross_validate(BASELINE, {"strategy": cfg["selection"]["baseline_strategy"]}, X_train, y_train)

metric = cfg["selection"]["metric"]
alpha = cfg["selection"]["alpha"]

# Corrected t-test for every pair (baseline included); held-out vs training size is 1/(k-1)
rows = []
for model_a, model_b in itertools.combinations(fold_scores, 2):
    mean_diff, t_stat, p_value = corrected_resampled_ttest(
        [fold[metric] for fold in fold_scores[model_a]],
        [fold[metric] for fold in fold_scores[model_b]],
        test_train_ratio=1 / (N_SPLITS - 1),
    )
    rows.append({"model_a": model_a, "model_b": model_b, "mean_diff": mean_diff, "t_stat": t_stat, "p_value": p_value})
significance = pd.DataFrame(rows)
# Many pairs are tested at once, so p-values are Holm-adjusted before deciding what is significant
significance["p_holm"] = holm_adjust(significance["p_value"])
significance["significant"] = significance["p_holm"] < alpha

fold_scores_table = pd.DataFrame([
    {"model": name, "repeat": repeat, "fold": fold, **scores}
    for name, folds in fold_scores.items()
    for scores, (repeat, fold) in zip(folds, fold_positions(len(folds)))
])
FOLD_SCORES_PATH.parent.mkdir(parents=True, exist_ok=True)
fold_scores_table.round(4).to_csv(FOLD_SCORES_PATH, index=False)
significance.round(4).to_csv(SIGNIFICANCE_PATH, index=False)

run = wandb.init(**WANDB_SETTINGS, name=f"comparison-{VERSION}", job_type="evaluate",
                 config={"metric": metric, "alpha": alpha, "cv_folds": N_SPLITS, "cv_repeats": N_REPEATS})
run.log({"comparison/significance": wandb.Table(dataframe=significance.round(4)),
         "comparison/fold_scores": wandb.Table(dataframe=fold_scores_table.round(4))})
run.finish()

print(f"{int(significance['significant'].sum())} of {len(significance)} pairs significant after Holm correction")
significance.round(4).sort_values("p_value")

**3.11 Select best model**

In [ ]:
# Selection rule: only declare a winner as far as the statistics allow
# 1. Eligible models beat the baseline significantly; if none do, every model stays in the pool
# 2. Models not significantly different from the top model (by mean) are tied with it
# 3. Among tied models, the simplest one wins (simplicity_order), favouring explainable, stable models
def compare_pair(model_a: str, model_b: str) -> tuple[float, bool]:
    """Mean difference (a - b) and significance flag for one pair, whichever order it was stored in."""
    row = significance[(significance["model_a"] == model_a) & (significance["model_b"] == model_b)]
    if not row.empty:
        return float(row["mean_diff"].iloc[0]), bool(row["significant"].iloc[0])
    row = significance[(significance["model_a"] == model_b) & (significance["model_b"] == model_a)]
    return -float(row["mean_diff"].iloc[0]), bool(row["significant"].iloc[0])


means = {name: float(np.mean([fold[metric] for fold in folds])) for name, folds in fold_scores.items()}
models = [m for m in means if m != BASELINE]

beats_baseline = []
for model in models:
    diff, significant = compare_pair(model, BASELINE)
    if diff > 0 and significant:
        beats_baseline.append(model)
pool = beats_baseline or models

top = max(pool, key=means.get)
tied = [m for m in pool if m == top or not compare_pair(m, top)[1]]

simplicity_order = cfg["selection"]["simplicity_order"]
best_name = min(tied, key=lambda m: (simplicity_order.index(m) if m in simplicity_order else len(simplicity_order), -means[m]))
best_version = str(comparison.loc[best_name, "version"])
is_significant = bool(beats_baseline)

# Round-trip YAML keeps the comments and key order of config.yml when writing it back
yaml_rt = YAML()
config_doc = yaml_rt.load(CONFIG_PATH)
config_doc["experiment"]["best_model"]["name"] = best_name
config_doc["experiment"]["best_model"]["version"] = best_version
config_doc["experiment"]["best_model"]["significant"] = is_significant
yaml_rt.dump(config_doc, CONFIG_PATH)

print(f"Beats baseline significantly: {beats_baseline or 'none'}")
print(f"Top by mean {metric}: {top} ({means[top]:.4f}); statistically tied with it: {tied}")
print(f"Selected {best_name} v{best_version} (significant={is_significant}), saved to config.yml")